# MLflow Session 2 – Tracking Runs, Metrics, Parameters, and Artifacts (Hands-on Notebook)

# 1. Conceptual Foundation

### MLflow Tracking — Introduction

In real-world machine learning, models are **not built once**.

They are:
- Trained repeatedly
- Tuned with different parameters
- Evaluated under different conditions


##### Core Problem

Without a tracking system:

- Metrics become unreliable
- Parameters are forgotten
- Code versions are unclear
- Results cannot be reproduced


### MLflow Tracking Solves This

MLflow introduces a structured way to log:

- **Metrics** → How well the model performs
- **Parameters** → How the model was configured
- **Artifacts** → Files, outputs, and code


### Fundamental Unit: RUN

A **Run** represents:

> One complete execution of a training process

Each run is:
- Fully tracked
- Fully reproducible
- Stored inside an **Experiment**


### Mental Model

Experiment

└── Run 1 (params, metrics, artifacts)

└── Run 2 (params, metrics, artifacts)

└── Run 3 (params, metrics, artifacts)

# 2. Environment Setup

In [ ]:
# Install dependencies
!pip install mlflow scikit-learn

In [ ]:
# Core imports
import mlflow

# ML utilities
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import os

# 3. Establishing the Experiment

### Why This Step Matters

MLflow requires an **active experiment** before logging begins.

This ensures:
- All runs are organized
- Results are easy to compare
- Nothing is lost

In [ ]:
mlflow.set_experiment("MLflow_Tracking")

# 4. Anatomy of a Run

### Starting a Run

When a run starts:

- It becomes the **active context**
- All logging operations attach to it
- MLflow records metadata automatically


### Important Concept: Active Run

At any moment:
> Only ONE run is active

All logs go there until:
- The run ends, or
- A new run starts

In [ ]:
run = mlflow.start_run()

print("Run ID:", run.info.run_id)
print("Artifact Location:", run.info.artifact_uri)

# 5. Understanding Run Metadata

### Run Metadata Explained:

##### - `run_id`
A unique identifier for the run  
→ Used to retrieve or reference it later

##### - `artifact_uri`
Location where files are stored  


### What is a URI?

A **URI (Uniform Resource Identifier)** defines where resources live.

MLflow uses URIs for:
- Artifacts
- Models
- Storage backends

Think of it as:
> “Where does this run live?”

# 6. Controlled Model Training

### Purpose

We now simulate a real ML workflow:
- Generate data
- Train model
- Evaluate performance

This gives us **meaningful data to track**

In [ ]:
# Create dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    random_state=42
)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

# Train
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

# 7. Logging Metrics (Performance)

### Metrics = Model Performance

Metrics answer:

> “How good is this model?”

Examples:
- Accuracy
- Loss
- Precision / Recall


### Logging Strategy

- Use `log_metric()` for single values
- Use `log_metrics()` for grouped logging

In [ ]:
mlflow.log_metric("accuracy", accuracy)

In [ ]:
mlflow.log_metrics({
    "accuracy": accuracy,
    "train_size": len(X_train),
    "test_size": len(X_test)
})

# 8. Logging Parameters (Configuration)

### Parameters = Model Configuration

Parameters answer:

> “How was this model built?”

Examples:
- Algorithm type
- Hyperparameters
- Data settings


### Why This Matters

Without parameters:
- Results are meaningless
- Experiments are not reproducible

##### Logging Strategy

- Use `log_param()` for single values
- Use `log_params()` for grouped logging

In [ ]:
mlflow.log_param("model_type", "LogisticRegression")

In [ ]:
mlflow.log_params({
    "n_features": 10,
    "test_split_ratio": 0.2,
    "random_state": 42
})

# 9. Logging Artifacts (Files & Outputs)

### Artifacts = Supporting Evidence

Artifacts answer:

> “What files support this run?”

Examples:
- Training scripts
- Output files
- Reports
- Model files


### Best Practice

Always log:
- Important outputs
- Anything needed for reproducibility

In [ ]:
# Create artifact
with open("run_summary.txt", "w") as f:
    f.write("Model: LogisticRegression\n")
    f.write(f"Accuracy: {accuracy}\n")

mlflow.log_artifact("run_summary.txt")

# 10. Logging Multiple Artifacts

### Directory Logging

When multiple files are generated:

→ Log entire directories

In [ ]:
os.makedirs("artifact_bundle", exist_ok=True)

with open("artifact_bundle/info.txt", "w") as f:
    f.write("Experiment bundle")

with open("artifact_bundle/metrics.txt", "w") as f:
    f.write(str(accuracy))

mlflow.log_artifacts("artifact_bundle")

# 11. Ending the Run

### Why End the Run?

Explicitly ending ensures:
- Clean lifecycle
- Proper logging completion

In [ ]:
mlflow.end_run()

# 12. Production-Grade Pattern (Best Practice)

### Recommended Pattern: Context Manager

This ensures:
- Automatic run closure
- Cleaner code
- Fewer errors

In [ ]:
with mlflow.start_run() as run:

    model = LogisticRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_param("model", "LogisticRegression")

    print("Run ID:", run.info.run_id)

# Launch the UI 

In [ ]:
import subprocess

subprocess.Popen(
    ["mlflow", "ui", "--port", "5000"],
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("MLflow UI started and will stay running on Windows!")
print("http://localhost:5000")

### Open the Interface

> http://localhost:5000


### What to Observe

##### Experiment View
- Compare multiple runs
- Sort by metrics

##### Run Detail View
- Metrics
- Parameters
- Artifacts
- Execution status


### Key Insight

MLflow UI transforms:
> Raw logs → Actionable insights

# 14. Deep Conceptual Summary

### Final Mental Model

##### - Experiment
Container for runs

##### - Run
Single training execution

##### - Logging
Capturing everything important:
- Metrics
- Parameters
- Artifacts


##### Full Workflow

1. Set experiment  
2. Start run  
3. Train model  
4. Log everything  
5. End run  


##### Core Principle

> If it is not logged, it does not exist.


##### Professional Insight

MLflow Tracking is not just a tool.

It is:

→ A discipline of reproducibility  
→ A system of record for ML  
→ A foundation for scalable ML engineering

# 15. Exercise — Thinking Like an ML Engineer

### Task:

Run multiple experiments where you vary:
- Model parameters
- Dataset size
- Random seed


### Log:

- At least 3 metrics
- At least 3 parameters
- At least 1 artifact


### Analyze:

1. Which run performed best?
2. Why?
3. Can you reproduce it exactly?


##### Goal

Move from:
“Running models”

To:
“Engineering experiments”

# Solution Overview

### Solution — Thinking Like an ML Engineer

In this solution, we move beyond single runs and:

- Design **controlled experiments**
- Systematically vary parameters
- Log everything required for reproducibility
- Analyze results like an ML engineer


##### What We Will Do

We will vary:

- Dataset size
- Random seed
- Model parameter (regularization strength)

We will log:

- Metrics (performance)
- Parameters (configuration)
- Artifacts (evidence)


##### Key Principle

> Experiments must be **systematic**, not random

# 15-1. Setup

In [ ]:
import mlflow
import os

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

mlflow.set_experiment("MLflow_Experiment_Engineering")

# 15-2. Run Multiple Experiments

In [ ]:
dataset_sizes = [500, 1000]
random_seeds = [42, 7]
regularization_values = [0.1, 1.0]

for size in dataset_sizes:
    for seed in random_seeds:
        for C in regularization_values:

            with mlflow.start_run():

                # -----------------------------
                # Data
                # -----------------------------
                X, y = make_classification(
                    n_samples=size,
                    n_features=10,
                    random_state=seed
                )

                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.2, random_state=seed
                )

                # -----------------------------
                # Model
                # -----------------------------
                model = LogisticRegression(C=C, max_iter=200)
                model.fit(X_train, y_train)

                predictions = model.predict(X_test)

                # -----------------------------
                # Metrics
                # -----------------------------
                acc = accuracy_score(y_test, predictions)
                prec = precision_score(y_test, predictions)
                rec = recall_score(y_test, predictions)

                # -----------------------------
                # Logging Metrics
                # -----------------------------
                mlflow.log_metrics({
                    "accuracy": acc,
                    "precision": prec,
                    "recall": rec
                })

                # -----------------------------
                # Logging Parameters
                # -----------------------------
                mlflow.log_params({
                    "dataset_size": size,
                    "random_seed": seed,
                    "regularization_C": C
                })

                # -----------------------------
                # Logging Artifact
                # -----------------------------
                os.makedirs("run_artifacts", exist_ok=True)

                artifact_path = f"run_artifacts/summary_{size}_{seed}_{C}.txt"

                with open(artifact_path, "w") as f:
                    f.write(f"Dataset size: {size}\n")
                    f.write(f"Seed: {seed}\n")
                    f.write(f"C: {C}\n")
                    f.write(f"Accuracy: {acc}\n")
                    f.write(f"Precision: {prec}\n")
                    f.write(f"Recall: {rec}\n")

                mlflow.log_artifact(artifact_path)

# 15-3. Inspect in UI

In [ ]:
import subprocess

subprocess.Popen(
    ["mlflow", "ui", "--port", "5000"],
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("MLflow UI started and will stay running on Windows!")
print("http://localhost:5000")

Open:

http://localhost:5000


## What to Do in UI

- Sort runs by **accuracy**
- Compare metrics across runs
- Click best run → inspect parameters + artifacts

# 15-4. Analysis — Model Answer

### Analysis — Model Answer

##### 1. Which run performed best?

The best run is the one with the **highest accuracy** (or best balance of metrics).

Typically:
- Larger dataset → better generalization
- Proper regularization → better performance


##### 2. Why did it perform best?

Because of a combination of:

- **Dataset size** → More data improves learning
- **Random seed** → Different data splits affect results
- **Regularization (C)** → Controls overfitting vs underfitting


##### Key Insight

> Performance is not random — it is a result of controlled factors

# 15-5. Reproducibility — Model Answer

### Reproducibility — Model Answer

#### 3. Can you reproduce it exactly?

Yes — if the following are logged:

- Dataset size
- Random seed
- Model parameters
- Code logic


##### Why This Works

Because MLflow captures:

- Parameters → configuration
- Metrics → results
- Artifacts → supporting evidence


##### Core Principle

> A good experiment is one you can **re-run and obtain the same result**

# 15-6. Deep Insight

### From Running Models → Engineering Experiments

##### Beginner Mindset

- “Let me try this model”
- “Let me change this parameter”


### Professional Mindset

- Define variables to test
- Control all other factors
- Log everything
- Compare systematically


### Final Insight

> ML is not about models  
> It is about **controlled experimentation**


### What You Just Did

You implemented:

- Experimental design
- Parameter tracking
- Metric comparison
- Reproducibility


This is the foundation of:
- MLOps
- Production ML systems
- Scalable experimentation